# Estructuras de datos en Python con datos geográficos

Los objetivos de aprendizaje de este capítulo son:

- Elegir la estructura (`list`, `tuple`, `set`, `dict`) más adecuada para cada tipo de dato geográfico.
- Acceder a estructuras anidadas y modificarlas, como una tabla de atributos representada con listas y diccionarios.
- Filtrar y transformar datos con comprensiones (el equivalente en Python de *Seleccionar por atributos* y de la *Calculadora de campos*).
- Comprender por qué copiar estructuras anidadas puede producir resultados inesperados.
- Leer y escribir archivos CSV y GeoJSON con los módulos `csv` y `json`, y resumir su contenido.

## Elegir la estructura adecuada

Una estructura de datos es una forma de organizar la información para poder usarla con eficacia. Python incluye las estructuras de datos vistas hasta ahora (`list`, `set`, `tuple` y `dict`).

La habilidad clave es elegir entre ellas según el dato que se quiere representar. Algunos ejemplos:

| Concepto SIG | Estructura | Ejemplo |
|---|---|---|
| Par de coordenadas | `tuple` | `(-3.70, 40.42)` |
| Códigos únicos (CRS, usos del suelo) | `set` | `{"EPSG:4326", "EPSG:25830"}` |
| Fila de una tabla de atributos (una entidad) | `dict` | `{"name": "Madrid", "population": 3300000}` |
| Tabla de atributos de una capa, cada fila un `dict` | `list` de `dict` | `[{...}, {...}]` |

Cuatro preguntas ayudan a decidir la estructura de datos:

1. ¿Importa el orden de los elementos?
2. ¿Se admiten elementos repetidos?
3. ¿Los datos van a cambiar?
4. ¿Se accede a ellos por posición o por clave?

In [ ]:
# Coordinate pair: the position of each value sets its meaning (lon, lat)
madrid = (-3.70, 40.42)

# Unique values: the duplicated code is discarded automatically
crs_codes = {"EPSG:4326", "EPSG:25830", "EPSG:4326", "EPSG:3857"}

print(madrid[0], madrid[1])
print(crs_codes)
print(len(crs_codes), "unique codes")

Las tuplas son **inmutables**: intentar modificar una produce un error (`TypeError`). Esto resulta útil para las coordenadas, ya que evita alterarlas por accidente.

In [2]:
madrid[0] = 0

TypeError: 'tuple' object does not support item assignment

## Una capa como lista de diccionarios

Una capa vectorial se puede representar como una **lista de diccionarios**: cada diccionario es una entidad (una fila de la tabla de atributos) y cada clave es un campo. Esta idea es la base de herramientas como pandas y GeoPandas.

In [ ]:
cities = [
    {"name": "Madrid", "region": "Comunidad de Madrid",
     "population": 3300000, "location": (-3.70, 40.42)},
    {"name": "Barcelona", "region": "Cataluña",
     "population": 1660000, "location": (2.17, 41.39)},
    {"name": "Valencia", "region": "Comunidad Valenciana",
     "population": 800000, "location": (-0.38, 39.47)},
    {"name": "Sevilla", "region": "Andalucía",
     "population": 685000, "location": (-5.98, 37.39)},
    {"name": "Zaragoza", "region": "Aragón",
     "population": 680000, "location": (-0.88, 41.65)},
    {"name": "Málaga", "region": "Andalucía",
     "population": 580000, "location": (-4.42, 36.72)},
    {"name": "Bilbao", "region": "País Vasco",
     "population": 345000, "location": (-2.93, 43.26)},
    {"name": "Girona", "region": "Cataluña",
     "population": 103000, "location": (2.82, 41.98)},
]

print(len(cities), "features")
for city in cities:
    print(city["name"], "-", city["population"])

## Extensión espacial (bounding box)

Combinando la comprensión de listas y la búsqueda por claves, se puede extraer la extensión espacial de las ciudades dentro de la tabla `cities`.

En la línea `for lon, lat in locations`, cada tupla `(lon, lat)` se **desempaqueta** en dos variables. Así se separan las longitudes de las latitudes y se obtiene la extensión de la capa.

In [ ]:
locations = [city["location"] for city in cities]

lons = [lon for lon, lat in locations]
lats = [lat for lon, lat in locations]

bbox = (min(lons), min(lats), max(lons), max(lats))
print("min_lon, min_lat, max_lon, max_lat =", bbox)

### Ejercicio 1

Con la lista `cities`, obtener mediante comprensiones de listas y de diccionarios (cuaderno `06_dictionaries.ipynb`):

1. Una lista con los nombres de las ciudades de menos de 500 000 habitantes.
2. Un diccionario que asocie cada nombre de ciudad con su población.
3. La extensión `(min_lon, min_lat, max_lon, max_lat)` de las ciudades de Andalucía.

## Mutabilidad y referencias

Con estructuras anidadas aparece un error muy habitual. En Python, `b = a` **no crea una copia**: ambos nombres apuntan al mismo objeto, y un cambio realizado a través de uno se ve en el otro.

In [ ]:
original = [{"name": "Madrid", "population": 3300000}]
alias = original

alias[0]["population"] = 0
print(original)   # the original also changed

En el cuaderno `03_list_tuples_sets.ipynb` se copió una lista con `[:]`. El método `.copy()` hace lo mismo y está disponible también en diccionarios y conjuntos: `cities.copy()` equivale a `cities[:]`.

Ambas formas crean una **copia superficial**: duplican la lista exterior, pero los diccionarios interiores siguen compartidos.

In [ ]:
original = [{"name": "Madrid", "population": 3300000}]
alias = original.copy()

alias[0]["population"] = 0
print(original)   # the original also changed

Para duplicar todos los niveles se usa la función `deepcopy()` del módulo `copy`:

In [ ]:
import copy

original = [{"name": "Madrid", "population": 3300000}]
shallow = original.copy()
deep = copy.deepcopy(original)

original[0]["population"] = 3400000

print("original:", original[0]["population"])   # 3400000
print("shallow: ", shallow[0]["population"])    # 3400000 (shared dictionary)
print("deep:    ", deep[0]["population"])        # 3300000 (independent copy)

## Velocidad de búsqueda: lista frente a conjunto

Comprobar si un valor está en una lista obliga a revisarla elemento a elemento. Los conjuntos y los diccionarios, en cambio, localizan el valor casi de forma directa. La diferencia es mínima con pocos datos, pero muy notable con millones de identificadores (parcelas, puntos de una nube LiDAR, etc.).

El siguiente ejemplo mide esa diferencia. Antes, algunas notas sobre el código:

- `1_000_000`: Python permite separar las cifras de un número con guiones bajos para facilitar su lectura. Equivale a `1000000`.
- `for _ in range(100)`: el guion bajo (`_`) es un nombre de variable válido que, por convención, indica que su valor no se va a usar. Aquí solo interesa repetir el bloque 100 veces.
- `target in ids_list`: la comprobación se realiza, pero su resultado (`True` o `False`) no se guarda, porque solo interesa medir cuánto tarda.
- `time.perf_counter()`, del módulo `time`, devuelve un instante en segundos. La diferencia entre dos llamadas es el tiempo transcurrido.

In [ ]:
import time

ids_list = list(range(1_000_000))
ids_set = set(ids_list)
target = 999_999          # worst case for the list: the last element

start = time.perf_counter()
for _ in range(100):
    target in ids_list
time_list = time.perf_counter() - start

start = time.perf_counter()
for _ in range(100):
    target in ids_set
time_set = time.perf_counter() - start

print(f"list: {time_list:.4f} s")
print(f"set:  {time_set:.6f} s")

## Leer y escribir datos con `csv` y `json`

### El módulo `csv`

En el cuaderno `08_import_open.ipynb` se leyó `weather_stations.csv` dividiendo cada línea con `.split(",")`. El módulo `csv` de la biblioteca estándar hace este trabajo y, además, resuelve casos que `.split()` no contempla, como valores que contienen comas escritos entre comillas (`"Zaragoza, Aeropuerto"`).

`csv.DictReader` recorre un archivo abierto y devuelve cada fila como un diccionario cuyas claves son los nombres de la cabecera. Con `list()` se reúnen todas las filas en una lista. Al abrir un archivo para el módulo `csv` se recomienda indicar `newline=""`, para que sea el módulo quien gestione los saltos de línea.

In [ ]:
import csv
import json
from pathlib import Path

data_dir = Path.cwd().parent / "data"
output_dir = Path.cwd().parent / "output"
output_dir.mkdir(exist_ok=True)

stations_path = data_dir / "weather_stations.csv"

with open(stations_path, encoding="utf-8", newline="") as file:
    stations = list(csv.DictReader(file))

print(stations[0])
print(type(stations[0]["elevation_m"]))

Cada fila es ya un diccionario, sin desempaquetar valores ni escribir las claves a mano. Sin embargo, **todos los valores siguen siendo `str`**: el módulo `csv` no convierte tipos. Los valores numéricos se convierten igual que en el cuaderno `08_import_open.ipynb`:

In [ ]:
for station in stations:
    station["latitude"] = float(station["latitude"])
    station["longitude"] = float(station["longitude"])
    station["elevation_m"] = int(station["elevation_m"])

print(stations[0])

### GeoJSON y el módulo `json`

**JSON** (*JavaScript Object Notation*) es un formato de texto muy utilizado para intercambiar datos. Su estructura coincide casi por completo con la de Python: los objetos JSON (entre llaves) equivalen a diccionarios y los *arrays* (entre corchetes), a listas.

**GeoJSON** es un formato JSON para datos geográficos, presentado en el cuaderno `06_dictionaries.ipynb`:

- Una capa es un diccionario con `"type": "FeatureCollection"` y una lista `"features"`.
- Cada entidad (`"Feature"`) tiene un diccionario `"properties"`, con sus atributos, y otro `"geometry"`, con el tipo de geometría (`"Point"`, `"LineString"`, `"Polygon"`, etc.) y sus coordenadas.
- Las coordenadas están en WGS 84, en grados decimales.

> **Atención al orden de las coordenadas.** GeoJSON escribe siempre **`[longitud, latitud]`**, es decir, `[x, y]`, y no `[latitud, longitud]`, como hacen muchos visores de mapas. Intercambiarlas es un error clásico: Zaragoza `(41.66, -1.01)` leída como longitud y latitud aparece cerca de la costa de Somalia.

El módulo `json` traduce entre texto JSON y estructuras de Python:

| Función | Uso |
| --- | --- |
| `json.load(file)` | Lee un archivo JSON abierto y devuelve diccionarios y listas |
| `json.dump(data, file)` | Escribe diccionarios y listas en un archivo abierto |

### Escribir un GeoJSON

Las estaciones leídas con `csv` se pueden convertir en una capa de puntos. Cada estación se transforma en una entidad: su nombre y su altitud pasan a `properties`, y su longitud y su latitud, **en este orden**, a `coordinates`:

In [ ]:
features = []
for station in stations:
    feature = {
        "type": "Feature",
        "properties": {
            "name": station["name"],
            "elevation_m": station["elevation_m"],
        },
        "geometry": {
            "type": "Point",
            "coordinates": [station["longitude"], station["latitude"]],
        },
    }
    features.append(feature)

stations_layer = {"type": "FeatureCollection", "features": features}
print(stations_layer["features"][0])

`json.dump()` guarda la capa en un archivo abierto en modo escritura. El argumento `indent=2` añade sangría para que el resultado sea legible, y `ensure_ascii=False` conserva caracteres como la `ñ` de Logroño en lugar de sustituirlos por códigos (`\u00f1`):

In [ ]:
geojson_path = output_dir / "weather_stations.geojson"

with open(geojson_path, "w", encoding="utf-8") as file:
    json.dump(stations_layer, file, indent=2, ensure_ascii=False)

print(geojson_path.read_text(encoding="utf-8")[:300])

El archivo `output/weather_stations.geojson` puede abrirse en QGIS (*Capa > Añadir capa > Añadir capa vectorial*, o arrastrándolo al mapa) para comprobar que cada estación aparece en su posición.

### Leer un GeoJSON

La carpeta `data` contiene dos capas reales del municipio de Belchite (Zaragoza), descargadas de [ICEARAGON](https://icearagon.aragon.es/) (Infraestructura de Conocimiento Espacial de Aragón):

- `icearagon_belchite_nomgeo.geojson`: topónimos (nombres geográficos) del mapa topográfico, como puntos.
- `icearagon_belchite_redviaria.geojson`: tramos de la red viaria, como líneas.

`json.load()` convierte el archivo en un diccionario:

In [ ]:
toponyms_path = data_dir / "icearagon_belchite_nomgeo.geojson"

with open(toponyms_path, encoding="utf-8") as file:
    toponyms = json.load(file)

print(type(toponyms))
print(toponyms.keys())
print(len(toponyms["features"]), "features")
print(toponyms["features"][0])

Además de `type` y `features`, el archivo incluye `name` y `crs`, añadidos por el programa que lo exportó. `CRS84` equivale a WGS 84 con el orden longitud, latitud.

Las coordenadas tienen un tercer valor, la altitud (`z`). En casi todas las entidades de esta capa vale `0.0`, lo que indica que no se ha registrado.

### Convertir la capa en una tabla

La estructura original es profunda (`feature["properties"]["NOMBRE"]`). Una comprensión permite convertirla en una tabla plana, es decir, una lista de diccionarios como `cities`. La longitud es el primer valor de `coordinates` y la latitud, el segundo:

In [ ]:
table = [
    {
        "name": feature["properties"]["NOMBRE"],
        "lon": feature["geometry"]["coordinates"][0],
        "lat": feature["geometry"]["coordinates"][1],
    }
    for feature in toponyms["features"]
]

for row in table[:5]:
    print(row)

### Agrupar y contar

Para contar entidades por categoría se usa un diccionario como contador. La expresión `counts.get(vehicle_type, 0)` devuelve el recuento actual, o `0` si la categoría aún no existe, y a ese valor se le suma 1.

El siguiente ejemplo cuenta los tramos de la red viaria según el tipo de vehículo que puede circular por ellos (campo `TIPO_VEHIC`):

In [ ]:
roads_path = data_dir / "icearagon_belchite_redviaria.geojson"

with open(roads_path, encoding="utf-8") as file:
    roads = json.load(file)

counts = {}
for feature in roads["features"]:
    vehicle_type = feature["properties"]["TIPO_VEHIC"]
    counts[vehicle_type] = counts.get(vehicle_type, 0) + 1

print(counts)

El resultado muestra un problema habitual en los datos reales: `"Peatón+bici+vehícu"` y `"Peatón+bici+vehículo"` representan la misma categoría, pero el primer texto aparece cortado. Además, `"Peaton+bici"` está escrito sin tilde. Antes de analizar unos datos conviene revisar sus valores únicos, por ejemplo, con un `set`.

### Ejercicio 2

Con la lista `table`, creada a partir de los topónimos:

1. Obtener, mediante una comprensión, los nombres de los topónimos que contienen la palabra `"Cabezo"`.
2. Calcular la extensión `(min_lon, min_lat, max_lon, max_lat)` de la capa, siguiendo el ejemplo de la sección *Extensión espacial (bounding box)*.

### Ejercicio 3

Con la capa `roads`:

1. Obtener los valores únicos del campo `TIPO_VEHIC`. *Pista:* `set()` convierte una lista en un conjunto.
2. El campo `VELOCIDAD` contiene texto. Obtener, mediante una comprensión, una lista con la velocidad de cada tramo como `int` y calcular la velocidad media.
3. Unificar las categorías de `TIPO_VEHIC` para que `"Peatón+bici+vehícu"` pase a ser `"Peatón+bici+vehículo"` y `"Peaton+bici"`, `"Peatón+bici"`. Volver a contar los tramos por categoría.

## Siguientes pasos

Las estructuras de este capítulo son la base de las bibliotecas que se usarán más adelante en el curso para gestionar datos geográficos de forma eficiente. Por ejemplo, `numpy` trabaja con matrices numéricas, como las celdas de un ráster, y `geopandas` lee directamente archivos GeoJSON o Shapefile como tablas con una columna de geometría, sin necesidad de recorrer diccionarios a mano.

## Resumen

- La estructura se elige según el dato: `tuple` para valores fijos como un par de coordenadas, `set` para valores únicos, `dict` para los atributos de una entidad y una `list` de `dict` para una tabla de atributos.
- Las comprensiones permiten filtrar y transformar tablas, como *Seleccionar por atributos* o la *Calculadora de campos* de un SIG.
- `b = a` no copia: ambos nombres apuntan al mismo objeto. `.copy()` y `[:]` crean una **copia superficial**, que comparte los elementos interiores; `copy.deepcopy()` duplica todos los niveles.
- Comprobar si un valor está en un `set` o en un `dict` es mucho más rápido que en una `list`.
- `csv.DictReader` lee cada fila de un CSV como un diccionario. Todos los valores se leen como `str`.
- GeoJSON organiza una capa en un `FeatureCollection` con una lista de entidades, cada una con `properties` y `geometry`. Las coordenadas se escriben como **`[longitud, latitud]`**.
- `json.load()` lee un archivo JSON como diccionarios y listas; `json.dump()` los escribe.
- Un diccionario con `.get(clave, 0) + 1` permite contar entidades por categoría.
- Los datos reales suelen contener errores, como categorías escritas de varias formas. Conviene revisar sus valores únicos antes de analizarlos.